<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook A01: Loading and Manipulating the Data</h2>
</div>

Worked solutions to the 4 exercises in
[Notebook A01: Loading and Manipulating the Data](../notebooks/A01_Loading_data.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The dataset and the long-format frame the first exercise builds on.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

df = pd.read_parquet(nb_config.CDC_TEMP_PATH)

df_long = pd.melt(
    df,
    ignore_index=False,
    value_vars=df.columns,
    var_name="region",
    value_name="temperature",
)
df_long.index.name = "date"
df_long = df_long.sort_index()

print(f"{df.shape[0]} months x {df.shape[1]} regions, "
      f"{df.index.min().date()} to {df.index.max().date()}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Convert `df_long` directly to compact format without going through wide first. Each group in `df_long` corresponds to one series.

In [ ]:
compact_from_long = df_long.groupby("region").apply(
    lambda group: pd.Series({
        "Start": group.index.min(),
        "Frequency": "1MS",
        "n_Elements": len(group),
        "Values": group["temperature"].to_numpy(),
    }),
    include_groups=False,
)

compact_from_long

`groupby("region")` splits the long frame into one group per series, and each group is already sorted by
date because the frame was. From there the four compact-format fields fall out directly: the first index
value, the frequency, the length, and the values as an array.

The one thing worth checking is that this agrees with the route through wide format.

In [ ]:
# The wide route, for comparison
compact_from_wide = pd.DataFrame(
    {
        "Start": df.apply(lambda column: column.first_valid_index()),
        "Frequency": "1MS",
        "n_Elements": df.shape[0],
        "Values": [df[column].values for column in df.columns],
    },
    index=df.columns,
)

# groupby sorts its keys, so put both frames in the same order before comparing
aligned = compact_from_long.reindex(compact_from_wide.index)

same_length = (aligned["n_Elements"] == compact_from_wide["n_Elements"]).all()
same_values = all(
    np.allclose(aligned.loc[name, "Values"], compact_from_wide.loc[name, "Values"])
    for name in compact_from_wide.index
)

print(f"Row order differs: {list(compact_from_long.index[:2])} "
      f"vs {list(compact_from_wide.index[:2])}")
print(f"Same lengths after aligning: {same_length}")
print(f"Same values after aligning:  {same_values}")

Both routes agree once the rows are put in the same order, and that caveat is the useful part of the
check. `groupby` returns its keys **sorted**, while the wide route preserves the column order of the
original frame. Comparing the two without aligning them raises `ValueError: Can only compare
identically-labeled Series objects`, which is pandas being helpful: silently comparing mismatched rows
would be far worse.

Which route you use is a question of where the data arrives from: long format is what a
database query or a CSV export usually gives you, and converting it directly avoids materialising a wide
frame that may be large and mostly empty when series cover different periods.

Note the `include_groups=False` argument. Recent pandas versions warn when the grouping column is passed
into the function alongside the rest, and this says explicitly that we do not want it.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Resample the CDC dataset to 10-year intervals and compute the maximum temperature recorded in each decade for `Deutschland`. Which decade was the warmest on record?

In [ ]:
by_decade = df["Deutschland"].resample("10YS").max()

by_decade.round(2)

**The decade beginning 2001 holds the record, at 21.99 °C.**

The number itself is the warmest single *month* in each decade, not a decade average, which is what the
exercise asked for and is worth being clear about: this is a statement about extremes, not about the
climate of the decade.

Two things in that output deserve a second look.

The trend is not monotonic. The 2011 and 2021 decades come in lower than 2001, at 20.29 and 20.23. A
record is a single observation, so it is noisy in a way a decade average is not; one exceptional month in
2003 sets a bar that a warmer decade can easily fail to clear.

And **the last bucket is not a decade**. The data ends in 2025, so the bucket beginning 2021 holds five
years rather than ten, and it is competing for a maximum with half the opportunities. Resampling silently
produces partial buckets at the end of a series, and comparing them with complete ones is an easy mistake
to make.

In [ ]:
# How many months went into each bucket?
counts = df["Deutschland"].resample("10YS").count()

pd.DataFrame({"max °C": by_decade.round(2), "months": counts}).tail(4)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Select all observations from January across all years (i.e. every row where the month is 1). What is the coldest January on record for `Deutschland`?

In [ ]:
januaries = df[df.index.month == 1]["Deutschland"]

print(f"{len(januaries)} Januaries, {januaries.index.min().year} to {januaries.index.max().year}")
print(f"\nColdest: {januaries.min():.2f} °C in {januaries.idxmin().year}")
print(f"Warmest: {januaries.max():.2f} °C in {januaries.idxmax().year}")
print("\nFive coldest:")
print(januaries.nsmallest(5).round(2).to_string())

**January 1940, at -9.0 °C**, and by a clear margin: the next coldest is 1942 at -7.9 °C.

The winters of 1940 and 1942 were exceptional across Europe, and they show up in this dataset as the two
coldest Januaries in 145 years of record. The gap between the coldest (-9.0) and the warmest (+4.8, in
2007) is nearly 14 °C, which is a useful thing to know before forecasting: January is not a temperature,
it is a distribution with a very long left tail.

`df.index.month == 1` produces a boolean array the same length as the frame, which is the general pattern
for selecting on a property of the index rather than a range of it. The same form works for
`index.dayofweek`, `index.quarter`, or any other calendar attribute.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-4">Exercise 4</h3>
</div>

> Compute a 12-month centred rolling mean for `Deutschland`. A centred window (`center=True`) places the window symmetrically around each observation rather than looking only backwards. Does the result look different from the standard rolling mean? What does centering trade off?

In [ ]:
germany = df["Deutschland"]

trailing = germany.rolling(12).mean()
centred = germany.rolling(12, center=True).mean()

print(f"trailing: first value {trailing.first_valid_index().date()}, "
      f"last {trailing.last_valid_index().date()}")
print(f"centred:  first value {centred.first_valid_index().date()}, "
      f"last {centred.last_valid_index().date()}")

# How far apart are the two curves, and is it a lag?
alignment = {
    shift: trailing.shift(shift).corr(centred) for shift in range(-12, 1)
}
best_shift = max(alignment, key=alignment.get)

print(f"\nCorrelation as they stand:     {trailing.corr(centred):.4f}")
print(f"Best alignment at a shift of:  {best_shift} months "
      f"(correlation {alignment[best_shift]:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

window = slice("1995", "2010")
ax.plot(germany[window], color="lightsteelblue", linewidth=0.8, label="Monthly")
ax.plot(trailing[window], color="crimson", linewidth=1.8, label="Trailing 12-month mean")
ax.plot(centred[window], color="seagreen", linewidth=1.8, label="Centred 12-month mean")

ax.set_title("Trailing and centred rolling means", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Yes, and the difference is a shift in time — nothing else.** Move the trailing mean back by five months
and the correlation with the centred mean is **exactly 1.0000**. They are not merely similar curves, they
are the same numbers plotted in different places.

The reason is arithmetic. A trailing window at month *t* averages months *t-11* to *t*, so its value
describes the middle of that stretch rather than its end. A centred 12-month window at *t* covers *t-6* to
*t+5* — pandas puts the extra observation on the right when the window is even, which is why the offset
comes out at five rather than six.

Centring corrects that, and the price is visible in the first cell's output. The trailing mean runs to
**August 2025**, the end of the data. The centred mean stops in **March 2025**, five months short, because
computing a centred value for August would need data from the following February.

That is the trade-off, and which side of it you want depends entirely on what the smoothing is for:

- **Describing the past**, in a chart or a report, wants the centred version. A trend line that lags its
  own data by six months is misleading, and losing the final few months costs nothing.
- **Forecasting** cannot use it at all. The centred mean at any recent point requires observations that
  have not happened yet, which is precisely the leakage Notebook
  [C01](../notebooks/C01_Feature_engineering.ipynb) is about. Every rolling feature in Parts C and D is
  trailing, and deliberately so.

The same distinction applies to any smoother: centring buys accuracy about *when* something happened, at
the cost of not being computable at the edge you care most about.

---

Back to [Notebook A01](../notebooks/A01_Loading_data.ipynb), or on to
[Notebook A02](../notebooks/A02_Basic_plotting.ipynb).